# RT-DETR-v2 Termal Veri Üzerinde Fine-tune (Google Colab)

Colab T4/A100 üzerinde çalışacak şekilde uyarlanmıştır. Çalıştırmadan önce:
1. **Runtime → Change runtime type → GPU** seç (T4 yeterli, A100/L4 hızlı).
2. Termal datasetini Google Drive'a yükle (örn. `MyDrive/thermal/dataset_augmented/`).
3. Hücreleri sırayla çalıştır.


In [ ]:
# Colab paket kurulumu (yerel ortamda zaten varsa atla)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q -U transformers accelerate albumentations torchmetrics pycocotools


In [ ]:
# Drive mount (sadece Colab'da)
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# GPU doğrulama
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('BF16 destek:', torch.cuda.is_bf16_supported())


In [ ]:
# (Opsiyonel) HF token — Hub'a push etmek istersen
# from huggingface_hub import login
# login()  # Settings > Access Tokens > write yetkili token


In [ ]:
checkpoint = "PekingU/rtdetr_v2_r18vd"
image_size = 320

In [ ]:
import json
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset as TorchBaseDataset

# Dataset konumu — Drive'da nereye yüklediğine göre güncelle
if IN_COLAB:
    DATASET_ROOT = Path("/content/drive/MyDrive/thermal/dataset_augmented")
else:
    DATASET_ROOT = Path("dataset/dataset_augmented")

assert DATASET_ROOT.exists(), f"Dataset bulunamadı: {DATASET_ROOT}"


class LocalCocoDataset(TorchBaseDataset):
    """COCO formatlı tek split için minimal wrapper.

    HF CPPE-5 örneğiyle aynı dict shape'i döndürür ki notebook'un geri
    kalanı (ThermalCocoDataset, görselleştirme, vb.) aynen çalışsın.
    """

    def __init__(self, split_dir):
        split_dir = Path(split_dir)
        self.images_dir = split_dir / "images"
        with open(split_dir / "_annotations.coco.json") as f:
            data = json.load(f)

        self.categories = sorted(data["categories"], key=lambda c: c["id"])
        self.images = data["images"]

        self._anns_by_img = {}
        for a in data["annotations"]:
            self._anns_by_img.setdefault(a["image_id"], []).append(a)

    @property
    def category_names(self):
        return [c["name"] for c in self.categories]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        info = self.images[idx]
        anns = self._anns_by_img.get(info["id"], [])
        image = Image.open(self.images_dir / info["file_name"]).convert("RGB")
        return {
            "image_id": info["id"],
            "image":    image,
            "width":    info["width"],
            "height":   info["height"],
            "objects": {
                "id":       [a["id"]          for a in anns],
                "area":     [a["area"]        for a in anns],
                "bbox":     [a["bbox"]        for a in anns],
                "category": [a["category_id"] for a in anns],
            },
        }


dataset = {
    "train":      LocalCocoDataset(DATASET_ROOT / "train"),
    "validation": LocalCocoDataset(DATASET_ROOT / "val"),
    "test":       LocalCocoDataset(DATASET_ROOT / "test"),
}

{k: len(v) for k, v in dataset.items()}


You'll see that this dataset has 1000 images for train and validation sets and a test set with 29 images.

To get familiar with the data, explore what the examples look like.

In [ ]:
dataset["train"][0]

The examples in the dataset have the following fields:
- `image_id`: the example image id
- `image`: a `PIL.Image.Image` object containing the image
- `width`: width of the image
- `height`: height of the image
- `objects`: a dictionary containing bounding box metadata for the objects in the image:
  - `id`: the annotation id
  - `area`: the area of the bounding box
  - `bbox`: the object's bounding box (in the [COCO format](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/#coco) )
  - `category`: the object's category, with possible values including `Coverall (0)`, `Face_Shield (1)`, `Gloves (2)`, `Goggles (3)` and `Mask (4)`

You may notice that the `bbox` field follows the COCO format, which is the format that the RT-DETRv2 model expects.
However, the grouping of the fields inside `objects` differs from the annotation format RT-DETRv2 requires. You will
need to apply some preprocessing transformations before using this data for training.

To get an even better understanding of the data, visualize an example in the dataset.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw

# Get mapping from category id to category name (from local COCO json)
categories = dataset["train"].category_names
id2label = {idx: name for idx, name in enumerate(categories)}
label2id = {v: k for k, v in id2label.items()}

# Load image and annotations
sample      = dataset["train"][2]
image       = sample["image"].copy()
annotations = sample["objects"]

# Draw bounding boxes and labels
draw = ImageDraw.Draw(image)
for i in range(len(annotations["id"])):
    box       = annotations["bbox"][i]
    class_idx = annotations["category"][i]
    x, y, w, h = tuple(box)
    draw.rectangle((x, y, x + w, y + h), outline="red", width=1)
    draw.text((x, y), id2label[class_idx], fill="white")

image

To visualize the bounding boxes with associated labels, you can get the labels from the dataset's metadata, specifically
the `category` field.
You'll also want to create dictionaries that map a label id to a label class (`id2label`) and the other way around (`label2id`).
You can use them later when setting up the model. Including these maps will make your model reusable by others if you share
it on the Hugging Face Hub. Please note that, the part of above code that draws the bounding boxes assume that it is in `COCO` format `(x_min, y_min, width, height)`. It has to be adjusted to work for other formats like `(x_min, y_min, x_max, y_max)`.

As a final step of getting familiar with the data, explore it for potential issues. One common problem with datasets for
object detection is bounding boxes that "stretch" beyond the edge of the image. Such "runaway" bounding boxes can raise
errors during training and should be addressed. There are a few examples with this issue in this dataset.
To keep things simple in this guide, we will set `clip=True` for `BboxParams` in transformations below.

## Preprocess the data

To finetune a model, you must preprocess the data you plan to use to match precisely the approach used for the pre-trained model.
[AutoImageProcessor](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoImageProcessor) takes care of processing image data to create `pixel_values`, `pixel_mask`, and
`labels` that a DETR model can train with. The image processor has some attributes that you won't have to worry about:

- `image_mean = [0.485, 0.456, 0.406 ]`
- `image_std = [0.229, 0.224, 0.225]`

These are the mean and standard deviation used to normalize images during the model pre-training. These values are crucial
to replicate when doing inference or finetuning a pre-trained image model.

Instantiate the image processor from the same checkpoint as the model you want to finetune.

In [ ]:
from transformers import AutoImageProcessor

image_processor = AutoImageProcessor.from_pretrained(
    checkpoint,
    do_resize=True,
    size={"width": image_size, "height": image_size},
    use_fast=True,
)

Before passing the images to the `image_processor`, apply two preprocessing transformations to the dataset:
- Augmenting images
- Reformatting annotations to meet RT-DETRv2 expectations

First, to make sure the model does not overfit on the training data, you can apply image augmentation with any data augmentation library. Here we use [Albumentations](https://albumentations.ai/docs/).
This library ensures that transformations affect the image and update the bounding boxes accordingly.
The 🤗 Datasets library documentation has a detailed [guide on how to augment images for object detection](https://huggingface.co/docs/datasets/object_detection),
and it uses the exact same dataset as an example. Apply the same approach here, resize each image,
flip it horizontally, and brighten it. For additional augmentation options, explore the [Albumentations Demo Space](https://huggingface.co/spaces/qubvel-hf/albumentations-demo).

In [ ]:
import albumentations as A

train_augmentation_and_transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
    ],
    bbox_params=A.BboxParams(format="coco", label_fields=["category"], clip=True, min_area=25, min_width=1, min_height=1),
)

validation_transform = A.Compose(
    [A.NoOp()],
    bbox_params=A.BboxParams(format="coco", label_fields=["category"], clip=True, min_area=1, min_width=1, min_height=1),
)

Visualize some augmented images to make sure images look natural and annotations are correct:

In [ ]:
for i in [15, 16, 17]:
    sample      = dataset["train"][i]
    image       = sample["image"]
    annotations = sample["objects"]

    # Apply the augmentation
    output = train_augmentation_and_transform(
        image=np.array(image), bboxes=annotations["bbox"], category=annotations["category"]
    )

    # Unpack the output
    image = Image.fromarray(output["image"])
    categories_aug, boxes = output["category"], output["bboxes"]

    # Draw the augmented image
    draw = ImageDraw.Draw(image)
    for category, box in zip(categories_aug, boxes):
        x, y, w, h = box
        draw.rectangle((x, y, x + w, y + h), outline="red", width=1)
        draw.text((x, y), id2label[category], fill="white")

    display(image.resize([256, 256]))

The `image_processor` expects the annotations to be in the following format: `{'image_id': int, 'annotations': List[Dict]}`,
 where each dictionary is a COCO object annotation. Let's add a function to reformat annotations for a single example:

In [ ]:
from torch.utils.data import Dataset


class ThermalCocoDataset(Dataset):
    def __init__(self, dataset, image_processor, transform=None):
        self.dataset = dataset
        self.image_processor = image_processor
        self.transform = transform

    @staticmethod
    def format_image_annotations_as_coco(image_id, categories, boxes):
        """Format one set of image annotations to the COCO format expected
        by the RT-DETR image processor.
        """
        annotations = []
        for category, bbox in zip(categories, boxes):
            annotations.append({
                "image_id":    image_id,
                "category_id": category,
                "bbox":        list(bbox),
                "iscrowd":     0,
                "area":        bbox[2] * bbox[3],
            })
        return {"image_id": image_id, "annotations": annotations}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]

        image_id   = sample["image_id"]
        image      = sample["image"]
        boxes      = sample["objects"]["bbox"]
        categories = sample["objects"]["category"]

        image = np.array(image.convert("RGB"))

        if self.transform:
            transformed = self.transform(image=image, bboxes=boxes, category=categories)
            image      = transformed["image"]
            boxes      = transformed["bboxes"]
            categories = transformed["category"]

        formatted_annotations = self.format_image_annotations_as_coco(image_id, categories, boxes)

        result = self.image_processor(
            images=image, annotations=formatted_annotations, return_tensors="pt"
        )

        result = {k: v[0] for k, v in result.items()}
        return result

Now you can combine the image and annotation transformations to use on a batch of examples:

In [ ]:
train_dataset      = ThermalCocoDataset(dataset["train"],      image_processor, transform=train_augmentation_and_transform)
validation_dataset = ThermalCocoDataset(dataset["validation"], image_processor, transform=validation_transform)
test_dataset       = ThermalCocoDataset(dataset["test"],       image_processor, transform=validation_transform)

train_dataset[15]

Apply this preprocessing function to the entire dataset using 🤗 Datasets [with_transform](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.Dataset.with_transform) method. This method applies
transformations on the fly when you load an element of the dataset.

At this point, you can check what an example from the dataset looks like after the transformations. You should see a tensor
with `pixel_values`, a tensor with `pixel_mask`, and `labels`.

Check images once again after applying the all the transformations, verify that boxes and labels are correct!

In [ ]:
for i in [15, 16, 17]:
    sample = train_dataset[i]

    # De-normalize image
    image = sample["pixel_values"]
    print("Image tensor shape:", image.shape)
    image = image.numpy().transpose(1, 2, 0)
    image = (image - image.min()) / (image.max() - image.min()) * 255.
    image = Image.fromarray(image.astype(np.uint8))

    # Convert boxes from [center_x, center_y, width, height] to [x, y, width, height] for visualization
    boxes = sample["labels"]["boxes"].numpy()
    print("Boxes shape:", boxes.shape)
    boxes[:, :2] = boxes[:, :2] - boxes[:, 2:] / 2
    w, h = image.size
    boxes = boxes * np.array([w, h, w, h])[None]

    categories = sample["labels"]["class_labels"].numpy()
    print("Categories shape:", categories.shape)

    # Draw boxes and labels on image
    draw = ImageDraw.Draw(image)
    for box, category in zip(boxes, categories):
        x, y, w, h = box
        draw.rectangle([x, y, x + w, y + h], outline="red", width=1)
        draw.text((x, y), id2label[category], fill="white")

    display(image)

You have successfully augmented the images and prepared their annotations. In the final step, create a custom `collate_fn` to batch images together.

In [ ]:
import torch

def collate_fn(batch):
    data = {}
    data["pixel_values"] = torch.stack([x["pixel_values"] for x in batch])
    data["labels"] = [x["labels"] for x in batch]
    return data

## Preparing function to compute mAP

Object detection models are commonly evaluated with a set of <a href="https://cocodataset.org/#detection-eval">COCO-style metrics</a>. We are going to use `torchmetrics` to compute `mAP` (mean average precision) and `mAR` (mean average recall) metrics and will wrap it to `compute_metrics` function in order to use in [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) for evaluation.

Intermediate format of boxes used for training is `YOLO` (normalized) but we will compute metrics for boxes in `Pascal VOC` (absolute) format in order to correctly handle box areas. Let's define a function that converts bounding boxes to `Pascal VOC` format:

Then, in `compute_metrics` function we collect `predicted` and `target` bounding boxes, scores and labels from evaluation loop results and pass it to the scoring function.

In [ ]:
import numpy as np
from dataclasses import dataclass
from transformers.image_transforms import center_to_corners_format
from torchmetrics.detection.mean_ap import MeanAveragePrecision


@dataclass
class ModelOutput:
    logits: torch.Tensor
    pred_boxes: torch.Tensor


class MAPEvaluator:

    def __init__(self, image_processor, threshold=0.00, id2label=None):
        self.image_processor = image_processor
        self.threshold = threshold
        self.id2label = id2label

    def collect_image_sizes(self, targets):
        """Collect image sizes across the dataset as list of tensors with shape [batch_size, 2]."""
        image_sizes = []
        for batch in targets:
            batch_image_sizes = torch.tensor(np.array([x["size"] for x in batch]))
            image_sizes.append(batch_image_sizes)
        return image_sizes

    def collect_targets(self, targets, image_sizes):
        post_processed_targets = []
        for target_batch, image_size_batch in zip(targets, image_sizes):
            for target, size in zip(target_batch, image_size_batch):

                # here we have "yolo" format (x_center, y_center, width, height) in relative coordinates 0..1
                # and we need to convert it to "pascal" format (x_min, y_min, x_max, y_max) in absolute coordinates
                height, width = size
                boxes = torch.tensor(target["boxes"])
                boxes = center_to_corners_format(boxes)
                boxes = boxes * torch.tensor([[width, height, width, height]])

                labels = torch.tensor(target["class_labels"])
                post_processed_targets.append({"boxes": boxes, "labels": labels})
        return post_processed_targets

    def collect_predictions(self, predictions, image_sizes):
        post_processed_predictions = []
        for batch, target_sizes in zip(predictions, image_sizes):
            batch_logits, batch_boxes = batch[1], batch[2]
            output = ModelOutput(logits=torch.tensor(batch_logits), pred_boxes=torch.tensor(batch_boxes))
            post_processed_output = self.image_processor.post_process_object_detection(
                output, threshold=self.threshold, target_sizes=target_sizes
            )
            post_processed_predictions.extend(post_processed_output)
        return post_processed_predictions

    @torch.no_grad()
    def __call__(self, evaluation_results):

        predictions, targets = evaluation_results.predictions, evaluation_results.label_ids

        image_sizes = self.collect_image_sizes(targets)
        post_processed_targets = self.collect_targets(targets, image_sizes)
        post_processed_predictions = self.collect_predictions(predictions, image_sizes)

        evaluator = MeanAveragePrecision(box_format="xyxy", class_metrics=True)
        evaluator.warn_on_many_detections = False
        evaluator.update(post_processed_predictions, post_processed_targets)

        metrics = evaluator.compute()

        # Replace list of per class metrics with separate metric for each class
        classes = metrics.pop("classes")
        map_per_class = metrics.pop("map_per_class")
        mar_100_per_class = metrics.pop("mar_100_per_class")
        for class_id, class_map, class_mar in zip(classes, map_per_class, mar_100_per_class):
            class_name = id2label[class_id.item()] if id2label is not None else class_id.item()
            metrics[f"map_{class_name}"] = class_map
            metrics[f"mar_100_{class_name}"] = class_mar

        metrics = {k: round(v.item(), 4) for k, v in metrics.items()}

        return metrics

eval_compute_metrics_fn = MAPEvaluator(image_processor=image_processor, threshold=0.01, id2label=id2label)

## Training the detection model

You have done most of the heavy lifting in the previous sections, so now you are ready to train your model!
The images in this dataset are still quite large, even after resizing. This means that finetuning this model will
require at least one GPU.

Training involves the following steps:
1. Load the model with [AutoModelForObjectDetection](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoModelForObjectDetection) using the same checkpoint as in the preprocessing.
2. Define your training hyperparameters in [TrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments).
3. Pass the training arguments to [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) along with the model, dataset, image processor, and data collator.
4. Call [train()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.train) to finetune your model.

When loading the model from the same checkpoint that you used for the preprocessing, remember to pass the `label2id`
and `id2label` maps that you created earlier from the dataset's metadata. Additionally, we specify `ignore_mismatched_sizes=True` to replace the existing classification head with a new one.

In [ ]:
from transformers import AutoModelForObjectDetection

model = AutoModelForObjectDetection.from_pretrained(
    checkpoint,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

In the [TrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments) use `output_dir` to specify where to save your model, then configure hyperparameters as you see fit. For `num_train_epochs=10` training will take about 15 minutes in Google Colab T4 GPU, increase the number of epoch to get better results.

Important notes:
 - Do not remove unused columns because this will drop the image column. Without the image column, you
can't create `pixel_values`. For this reason, set `remove_unused_columns` to `False`.
 - Set `eval_do_concat_batches=False` to get proper evaluation results. Images have different number of target boxes, if batches are concatenated we will not be able to determine which boxes belongs to particular image.

If you wish to share your model by pushing to the Hub, set `push_to_hub` to `True` (you must be signed in to Hugging
Face to upload your model).

In [ ]:
from transformers import TrainingArguments

# Çıktı dizini Drive'a yazılırsa Colab oturumu kapansa bile checkpointler korunur
if IN_COLAB:
    OUTPUT_DIR = "/content/drive/MyDrive/thermal/rtdetr_v2_r18vd_thermal_augmented"
else:
    OUTPUT_DIR = "rtdetr_v2_r18vd_thermal_augmented"

# T4: bf16 desteklemez → fp16 fallback. A100/L4: bf16 OK.
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=20,            # Colab session limiti için düşürüldü; uzun eğitim için artır
    max_grad_norm=0.1,
    learning_rate=5e-5,
    warmup_steps=300,
    per_device_train_batch_size=8,  # T4 16 GB için güvenli; A100'de 16+ yapabilirsin
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,  # effective batch ~16
    dataloader_num_workers=2,       # Colab CPU sınırlı
    dataloader_pin_memory=True,
    bf16=use_bf16,
    fp16=use_fp16,
    metric_for_best_model="eval_map",
    greater_is_better=True,
    load_best_model_at_end=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    remove_unused_columns=False,
    eval_do_concat_batches=False,
    report_to="none",
    push_to_hub=False,              # Hub'a göndermek için True yap + login()
)


Finally, bring everything together, and call [train()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.train):

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=image_processor,
    data_collator=collate_fn,
    compute_metrics=eval_compute_metrics_fn,
)

trainer.train()

## Evaluate

In [ ]:
from pprint import pprint

metrics = trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="eval")
pprint(metrics)

If you have set `push_to_hub` to `True` in the `training_args`, and you're authenticated with your Hugging Face token, the training checkpoints are pushed to the
Hugging Face Hub. Upon training completion, push the final model to the Hub as well by calling the [push_to_hub()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.push_to_hub) method.

In [ ]:
trainer.push_to_hub()

These results can be further improved by adjusting the hyperparameters in [TrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments). Give it a go!

## Inference

Now that you have finetuned a model, evaluated it, and uploaded it to the Hugging Face Hub, you can use it for inference.

In [ ]:
import torch
from pathlib import Path
from PIL import Image, ImageDraw

device = "cuda" if torch.cuda.is_available() else "cpu"

# Pick one test image from the thermal dataset
test_images_dir = DATASET_ROOT / "test" / "images"
sample_path = next(test_images_dir.glob("*.jpg"))
image = Image.open(sample_path).convert("RGB")
print("Using:", sample_path.name)
image

Load model and image processor from the Hugging Face Hub (skip to use already trained in this session):

In [ ]:
from pathlib import Path
from transformers import AutoImageProcessor, AutoModelForObjectDetection

# Eğitim çıktı dizininden en son checkpoint
output_dir = Path(OUTPUT_DIR)
checkpoints = sorted(output_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
if not checkpoints:
    raise FileNotFoundError(f"No checkpoint-* dirs under {output_dir.resolve()}")
model_dir = checkpoints[-1]
print("Loading:", model_dir)

image_processor = AutoImageProcessor.from_pretrained(model_dir)
model           = AutoModelForObjectDetection.from_pretrained(model_dir).to(device)


And detect bounding boxes:

In [ ]:
inputs = image_processor(images=[image], return_tensors="pt")
inputs = inputs.to(device)
with torch.no_grad():
    outputs = model(**inputs)
target_sizes = torch.tensor([image.size[::-1]])

result = image_processor.post_process_object_detection(outputs, threshold=0.4, target_sizes=target_sizes)[0]

for score, label, box in zip(result["scores"], result["labels"], result["boxes"]):
    box = [round(i, 2) for i in box.tolist()]
    print(
        f"Detected {model.config.id2label[label.item()]} with confidence "
        f"{round(score.item(), 3)} at location {box}"
    )

Let's plot the result:

In [ ]:
image_with_boxes = image.copy()
draw = ImageDraw.Draw(image_with_boxes)

for score, label, box in zip(result["scores"], result["labels"], result["boxes"]):
    box = [round(i, 2) for i in box.tolist()]
    x, y, x2, y2 = tuple(box)
    draw.rectangle((x, y, x2, y2), outline="red", width=1)
    text_label = model.config.id2label[label.item()]
    draw.text((x, y), f"{text_label} [ {score.item():.2f} ]", fill="blue")

image_with_boxes